# Chapter 03 — Your first applications: root window, lifecycle, and shutdown

**Book role:** Foundations  
**Suggested book space:** about 7 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Create and configure the root window.
- Explain the create-layout-mainloop lifecycle.
- Use a predictable shutdown path and window protocol.
- Build a small application using a function instead of top-level sprawl.

## 1. The application lifecycle

A conventional desktop app creates one `Tk()` root, constructs widgets, lays them out, connects callbacks, performs any lightweight initialization, and calls `mainloop()`. Multiple independent `Tk()` instances in one process are usually a mistake because each owns an interpreter and event-loop resources. Use `Toplevel` for secondary windows such as preferences or record details.

In a notebook, GUI examples need extra discipline: run only the cell that opens the window, then close it before moving on. In a packaged application, a normal `if __name__ == "__main__":` guard prevents the GUI from launching just because another module imports the application class.

### Worked example 1: A clean entry point

This design separates application construction from launching the event loop. It can be imported for testing without opening a window.

In [ ]:
import tkinter as tk
from tkinter import ttk


def build_app(root: tk.Tk) -> None:
    root.title("First analytics app")
    root.geometry("520x260")
    root.minsize(420, 220)

    frame = ttk.Frame(root, padding=24)
    frame.pack(fill="both", expand=True)
    ttk.Label(frame, text="Tkinter for data analytics", font=("TkDefaultFont", 16, "bold")).pack(anchor="w")
    ttk.Label(frame, text="Start with a small, responsive window.").pack(anchor="w", pady=(8, 16))
    ttk.Button(frame, text="Close", command=root.destroy).pack(anchor="e")


def main() -> None:
    root = tk.Tk()
    build_app(root)
    root.mainloop()


if __name__ == "__main__":
    main()

## 2. Geometry, resizing, and the window manager

`title()`, `geometry()`, `minsize()`, and `resizable()` communicate presentation and resize policy. Geometry strings use a form like `800x500`; they are not a substitute for responsive layout. Prefer a geometry manager plus frames, `fill`, `expand`, and grid weights so the application can adapt to different window sizes and font scaling.

Do not assume a fixed pixel layout will work on every platform or display scale. Use meaningful padding, test a narrow window, test keyboard navigation, and avoid truncating important values in analytics tables. The window manager also sends a close request; applications may bind `WM_DELETE_WINDOW` to a cleanup method when they own files, database connections, or worker threads.

### Worked example 2: Handle the window close protocol

This example demonstrates the protocol hook. In production, keep the close handler brief and put deterministic resource cleanup into a method that is safe to call once.

In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk

root = tk.Tk()
root.title("Close protocol")
root.geometry("330x140")


def request_close() -> None:
    if messagebox.askyesno("Exit", "Close this window?"):
        root.destroy()

root.protocol("WM_DELETE_WINDOW", request_close)
ttk.Label(root, text="Try the window close button.", padding=18).pack()
root.mainloop()

## 3. Functions and classes as application boundaries

A tiny demo can use module-level code, but a chapter-sized application benefits from a `main()` function. It provides one visible entry point and makes imports safe. A class becomes helpful when widgets share state and methods. Do not use classes merely to wrap a few lines; use them when they represent a window, controller, view, or object with a lifecycle.

Shutdown should be explicit: stop scheduled work, signal workers if applicable, close database resources, and destroy the root window. Avoid forcibly killing worker threads. A cancellation event or task-specific cleanup is safer than assuming that closing the window instantly stops all Python work.

## Guided lab: Lab: launchable application shell

Create `app.py` with a `main()` entry point, one root window, a header, a central content frame, a status label, and an Exit menu item. Set a minimum size and implement one shutdown path. Confirm that importing `app` from another module does not open the window.

## Review questions and practice

1. Why is `Toplevel` normally preferable to creating a second `Tk()`?
2. What is the purpose of an entry-point guard?
3. When should a window override `WM_DELETE_WINDOW`?
4. Why should window size not be the only layout strategy?

## Further reading

- Tk root and Toplevel: https://docs.python.org/3/library/tkinter.html#tkinter.Tk
- Python modules: https://docs.python.org/3/tutorial/modules.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.